# Installs

After running the cell below, a pop-up will appear suggesting to restart the session. You should do it. After that, jump to the [Imports](#Imports) section and start running the code cells from there on.

In [ ]:
!pip install llama-index llama-index-llms-together llama-index-utils-workflow

# Imports

In [ ]:
from llama_index.core.agent.workflow import ReActAgent, AgentWorkflow
from llama_index.llms.together import TogetherLLM
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import AgentStream, ToolCallResult

import json
from typing import Dict, Any

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Load Model

In [ ]:
llm = TogetherLLM(
    model="Qwen/Qwen2.5-7B-Instruct-Turbo", api_key="tgp_v1_j3Xi83j-kqONYGQ8V6aytuRKr78u89x4Hcb3cSH46nE"
)

# test it out
print(llm.complete("What's 1+1?").text)

# Define functions for the Model to use

In [ ]:
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# Define the Agent

In [ ]:
agent = ReActAgent(
    tools=[add,multiply],
    llm=llm,
)

# to store conversation history and state
ctx = Context(agent)

# Testing our Agent

In [ ]:
handler = agent.run("Can you add 5 and 3?", ctx=ctx)

response = await handler

print(str(response))

# What is happening under the hood?

Let's run step by step and check the intermediary outcomes

In [ ]:
async def check_step_by_step(agent, query):

  ctx = Context(agent)
  handler = agent.run(query, ctx=ctx)

  async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
      print(f"{ev.delta}", end="", flush=True)


In [ ]:
await check_step_by_step(agent, query="Can you add 5 and 3?")

# Agent Prompts

In [ ]:
print(agent.get_prompts()['react_header'].template)

# Tool calls

In [ ]:
print('The agent did {} calls to tools'.format(len(response.tool_calls)))

In [ ]:
# we can check the metadata about the tool calling performed
response.tool_calls

In [ ]:
# what tool was called?
response.tool_calls[0].tool_name

In [ ]:
# what arguments were passed to the tool?
response.tool_calls[0].tool_kwargs

In [ ]:
# What was the output of the tool?
response.tool_calls[0].tool_output.raw_output

# A few more examples

In [ ]:
await check_step_by_step(agent, query="3+5+10")

In [ ]:
await check_step_by_step(agent, query="20*5+10")

In [ ]:
# how can we improve this?
# await ....

# We can have more complex agents

In [ ]:
import requests
import xml.etree.ElementTree as ET
from typing import List, Dict, Optional
from datetime import datetime, timedelta
import time

class MedicalLiteratureAgent:
    """
    A LlamaIndex-compatible agent for searching medical literature using PubMed API
    """

    def __init__(self):
        self.base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
        self.email = "catolica_class25@example.com"  # Required by NCBI for API usage

    def search_pubmed(self, query: str, max_results: int = 10,
                     recent_only: bool = False) -> List[Dict]:
        """
        Search PubMed for medical literature

        Args:
            query: Search terms (e.g., "resistant hypertension elderly")
            max_results: Maximum number of results to return
            recent_only: If True, only return papers from last 2 years

        Returns:
            List of papers with title, authors, abstract, publication date, PMID
        """
        try:
            # Step 1: Search for PMIDs
            pmids = self._search_pmids(query, max_results, recent_only)

            if not pmids:
                return []

            # Step 2: Fetch detailed information for each PMID
            papers = self._fetch_paper_details(pmids)

            return papers

        except Exception as e:
            print(f"Error searching PubMed: {str(e)}")
            return []

    def search_clinical_reviews(self, condition: str, max_results: int = 5) -> List[Dict]:
        """
        Search specifically for clinical reviews and meta-analyses

        Args:
            condition: Medical condition to search for
            max_results: Maximum number of results

        Returns:
            List of high-evidence publications
        """
        # Add filters for systematic reviews and meta-analyses
        query = f"{condition} AND (systematic review[pt] OR meta-analysis[pt])"
        return self.search_pubmed(query, max_results, recent_only=True)

    def search_treatment_studies(self, condition: str, treatment: str,
                               max_results: int = 10) -> List[Dict]:
        """
        Search for treatment studies for a specific condition

        Args:
            condition: Medical condition
            treatment: Treatment/intervention
            max_results: Maximum number of results

        Returns:
            List of treatment-related studies
        """
        query = f"{condition} AND {treatment} AND (clinical trial[pt] OR randomized controlled trial[pt])"
        return self.search_pubmed(query, max_results, recent_only=True)

    def _search_pmids(self, query: str, max_results: int, recent_only: bool) -> List[str]:
        """Search for PMIDs using ESearch"""
        url = f"{self.base_url}esearch.fcgi"

        params = {
            'db': 'pubmed',
            'term': query,
            'retmax': max_results,
            'retmode': 'xml',
            'email': self.email
        }

        # Add date filter for recent papers
        if recent_only:
            two_years_ago = datetime.now() - timedelta(days=730)
            date_filter = two_years_ago.strftime("%Y/%m/%d")
            params['datetype'] = 'pdat'
            params['mindate'] = date_filter

        response = requests.get(url, params=params)
        response.raise_for_status()

        # Parse XML response
        root = ET.fromstring(response.content)
        pmids = []

        for id_elem in root.findall('.//Id'):
            pmids.append(id_elem.text)

        return pmids

    def _fetch_paper_details(self, pmids: List[str]) -> List[Dict]:
        """Fetch detailed information for given PMIDs using EFetch"""
        if not pmids:
            return []

        url = f"{self.base_url}efetch.fcgi"

        params = {
            'db': 'pubmed',
            'id': ','.join(pmids),
            'retmode': 'xml',
            'email': self.email
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        # Parse XML response
        root = ET.fromstring(response.content)
        papers = []

        for article in root.findall('.//PubmedArticle'):
            paper_info = self._parse_article(article)
            if paper_info:
                papers.append(paper_info)

        return papers

    def _parse_article(self, article) -> Optional[Dict]:
        """Parse individual article XML to extract relevant information"""
        try:
            # Extract PMID
            pmid_elem = article.find('.//PMID')
            pmid = pmid_elem.text if pmid_elem is not None else "Unknown"

            # Extract title
            title_elem = article.find('.//ArticleTitle')
            title = title_elem.text if title_elem is not None else "No title available"

            # Extract authors
            authors = []
            for author in article.findall('.//Author'):
                last_name = author.find('.//LastName')
                first_name = author.find('.//ForeName')
                if last_name is not None and first_name is not None:
                    authors.append(f"{first_name.text} {last_name.text}")

            # Extract abstract
            abstract_elem = article.find('.//AbstractText')
            abstract = abstract_elem.text if abstract_elem is not None else "No abstract available"

            # Extract publication date
            pub_date = self._extract_pub_date(article)

            # Extract journal
            journal_elem = article.find('.//Journal/Title')
            journal = journal_elem.text if journal_elem is not None else "Unknown journal"

            # Extract DOI
            doi_elem = article.find('.//ELocationID[@EIdType="doi"]')
            doi = doi_elem.text if doi_elem is not None else None

            return {
                'pmid': pmid,
                'title': title,
                'authors': authors[:3],  # Limit to first 3 authors
                'abstract': abstract[:500] + "..." if len(abstract) > 500 else abstract,
                'publication_date': pub_date,
                'journal': journal,
                'doi': doi,
                'pubmed_url': f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            }

        except Exception as e:
            print(f"Error parsing article: {str(e)}")
            return None

    def _extract_pub_date(self, article) -> str:
        """Extract publication date from article XML"""
        try:
            pub_date = article.find('.//PubDate')
            if pub_date is not None:
                year = pub_date.find('.//Year')
                month = pub_date.find('.//Month')
                day = pub_date.find('.//Day')

                date_parts = []
                if year is not None:
                    date_parts.append(year.text)
                if month is not None:
                    date_parts.append(month.text)
                if day is not None:
                    date_parts.append(day.text)

                return " ".join(date_parts)
        except:
            pass

        return "Unknown date"

def create_llamaindex_tools():
    from llama_index.core.tools import FunctionTool

    agent = MedicalLiteratureAgent()

    # Create LlamaIndex tools
    search_tool = FunctionTool.from_defaults(
        fn=agent.search_pubmed,
        name="search_pubmed",
        description="Search PubMed for medical literature. Use this when you need to find recent research papers on medical topics."
    )

    review_tool = FunctionTool.from_defaults(
        fn=agent.search_clinical_reviews,
        name="search_clinical_reviews",
        description="Search for systematic reviews and meta-analyses on medical conditions. Use this for high-quality evidence."
    )

    treatment_tool = FunctionTool.from_defaults(
        fn=agent.search_treatment_studies,
        name="search_treatment_studies",
        description="Search for clinical trials and treatment studies for specific conditions and interventions."
    )

    return [search_tool, review_tool, treatment_tool]


In [ ]:
tools = create_llamaindex_tools()

In [ ]:
agent = ReActAgent(
    tools=create_llamaindex_tools(),
    llm=llm,
)

# to store conversation history and state
ctx = Context(agent)

In [ ]:
query = "I'm looking for recent papers on 'resistant hypertension elderly'."
await check_step_by_step(agent, query=query)

In [ ]:
await check_step_by_step(agent, query=query + 'Give me the paper title, author names and date of each paper')

# Exercises - Create your own Agents!

## 1. Drug Interactions

Create an agent that can check for interactions between medications from a database

In [ ]:
DRUG_INTERACTIONS = {
    # Anticoagulants with other medications
    ("warfarin", "aspirin"): {
        "severity": "major",
        "description": "Increased risk of bleeding due to additive anticoagulant effects",
        "mechanism": "Pharmacodynamic interaction - both affect hemostasis",
        "clinical_action": "Monitor INR closely, consider dose adjustment, watch for bleeding signs"
    },

    ("warfarin", "amiodarone"): {
        "severity": "major",
        "description": "Amiodarone inhibits warfarin metabolism, increasing anticoagulation",
        "mechanism": "CYP2C9 inhibition",
        "clinical_action": "Reduce warfarin dose by 25-50%, monitor INR weekly initially"
    },

    # Cardiovascular medications
    ("atenolol", "verapamil"): {
        "severity": "major",
        "description": "Risk of severe bradycardia and heart block",
        "mechanism": "Additive effects on cardiac conduction",
        "clinical_action": "Avoid combination if possible, monitor heart rate and rhythm"
    },

    ("lisinopril", "spironolactone"): {
        "severity": "moderate",
        "description": "Increased risk of hyperkalemia",
        "mechanism": "ACE inhibitor reduces aldosterone, spironolactone blocks aldosterone",
        "clinical_action": "Monitor serum potassium levels regularly"
    },

    # Antibiotics and other medications
    ("ciprofloxacin", "theophylline"): {
        "severity": "major",
        "description": "Ciprofloxacin increases theophylline levels, risk of toxicity",
        "mechanism": "CYP1A2 inhibition",
        "clinical_action": "Monitor theophylline levels, consider dose reduction"
    },

    ("erythromycin", "simvastatin"): {
        "severity": "major",
        "description": "Increased risk of myopathy and rhabdomyolysis",
        "mechanism": "CYP3A4 inhibition increases statin levels",
        "clinical_action": "Avoid combination, use alternative antibiotic or statin"
    },

    # Psychiatric medications
    ("fluoxetine", "tramadol"): {
        "severity": "major",
        "description": "Increased risk of serotonin syndrome",
        "mechanism": "Excessive serotonergic activity",
        "clinical_action": "Monitor for serotonin syndrome symptoms, consider alternatives"
    },

    ("lithium", "hydrochlorothiazide"): {
        "severity": "major",
        "description": "Thiazide diuretics increase lithium levels and toxicity risk",
        "mechanism": "Reduced renal lithium clearance",
        "clinical_action": "Monitor lithium levels closely, consider dose adjustment"
    },

    # Diabetes medications
    ("metformin", "contrast_dye"): {
        "severity": "major",
        "description": "Risk of lactic acidosis in patients with renal impairment",
        "mechanism": "Contrast may cause acute kidney injury, reducing metformin clearance",
        "clinical_action": "Hold metformin 48h before and after contrast, check renal function"
    },

    ("glyburide", "fluconazole"): {
        "severity": "moderate",
        "description": "Increased risk of hypoglycemia",
        "mechanism": "CYP2C9 inhibition increases sulfonylurea levels",
        "clinical_action": "Monitor blood glucose closely, consider dose reduction"
    },

    # Pain medications
    ("morphine", "lorazepam"): {
        "severity": "major",
        "description": "Increased risk of respiratory depression and sedation",
        "mechanism": "Additive CNS depressant effects",
        "clinical_action": "Use lowest effective doses, monitor respiratory status"
    },

    ("ibuprofen", "lisinopril"): {
        "severity": "moderate",
        "description": "NSAIDs may reduce ACE inhibitor effectiveness and increase nephrotoxicity",
        "mechanism": "Prostaglandin inhibition affects renal function",
        "clinical_action": "Monitor blood pressure and renal function"
    },

    # Gastrointestinal medications
    ("omeprazole", "clopidogrel"): {
        "severity": "major",
        "description": "Reduced clopidogrel effectiveness",
        "mechanism": "CYP2C19 inhibition reduces clopidogrel activation",
        "clinical_action": "Use alternative PPI (pantoprazole) or H2 blocker"
    },

    ("simethicone", "levothyroxine"): {
        "severity": "moderate",
        "description": "Reduced levothyroxine absorption",
        "mechanism": "Binding in GI tract",
        "clinical_action": "Separate administration by 4+ hours"
    },

    # Miscellaneous important interactions
    ("allopurinol", "azathioprine"): {
        "severity": "major",
        "description": "Increased risk of bone marrow suppression",
        "mechanism": "Xanthine oxidase inhibition reduces azathioprine metabolism",
        "clinical_action": "Reduce azathioprine dose by 65-75%, monitor CBC"
    },

    ("cyclosporine", "grapefruit_juice"): {
        "severity": "moderate",
        "description": "Increased cyclosporine levels and toxicity risk",
        "mechanism": "CYP3A4 inhibition in gut wall",
        "clinical_action": "Avoid grapefruit juice, monitor cyclosporine levels"
    }
}

In [ ]:
def check_drug_interaction():
  # IMPLEMENT THIS FUNCTION

agent = ReActAgent(
    tools=[check_drug_interaction],
    llm=llm,
)

In [ ]:
# test your agent
query = """
I'm considering prescribing glyburide to my patient.

Here's the relevant patient history, check if there is any potential drug interaction with the current medication and glyburide.

Patient: Sarah Johnson
DOB: March 15, 1978
MRN: 12345678
Date: July 15, 2025

CHIEF COMPLAINT
"I'm having burning when I urinate and some vaginal discharge."
HISTORY OF PRESENT ILLNESS
45-year-old female presents with a 3-day history of dysuria and vaginal discharge. Patient describes the discharge as thick, white, and "cottage cheese-like." She reports mild vulvar itching and burning sensation during urination. No fever, chills, or abdominal pain. Patient completed a course of antibiotics for UTI two weeks ago. No recent sexual activity changes. LMP was 2 weeks ago, regular cycle.
PAST MEDICAL HISTORY

Type 2 Diabetes Mellitus (well-controlled)
Hypertension
Recurrent UTIs

CURRENT MEDICATIONS

Metformin 500mg twice daily
Lisinopril 10mg once daily
Fluconazole 150mg - single dose prescribed today
Multivitamin once daily

ALLERGIES
NKDA (No Known Drug Allergies)
PHYSICAL EXAMINATION
Vital Signs: BP 128/82, HR 76, Temp 98.4°F, RR 16
General: Alert, oriented, no acute distress
Pelvic: External genitalia with mild erythema, thick white discharge noted. No cervical motion tenderness.
ASSESSMENT AND PLAN
Primary Diagnosis: Vaginal candidiasis (thrush)
Plan:

Fluconazole 150mg single dose (already dispensed)
Patient education on hygiene measures
Avoid tight-fitting clothing
Follow up if symptoms persist beyond 7 days
Consider diabetes management review given recurrent infections

Follow-up: PRN or if symptoms worsen/persist

"""
await check_step_by_step(agent, query=query)

## Drug contra-indication

You now want to create a tool that checks for the contra-indications of a specific medication based on a database

In [ ]:
DRUG_CONTRAINDICATIONS = {
    # Cardiovascular medications
    "metoprolol": {
        "absolute": [
            {
                "condition": "severe_bradycardia",
                "description": "Heart rate < 50 bpm",
                "reason": "Beta-blockers further reduce heart rate",
                "severity": "absolute"
            },
            {
                "condition": "cardiogenic_shock",
                "description": "Acute heart failure with hypotension",
                "reason": "Negative inotropic effects worsen cardiac output",
                "severity": "absolute"
            },
            {
                "condition": "second_degree_av_block",
                "description": "Type II or complete heart block without pacemaker",
                "reason": "Risk of complete heart block",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "asthma",
                "description": "Active bronchospastic disease",
                "reason": "Non-selective beta blockade may cause bronchoconstriction",
                "severity": "relative",
                "alternatives": "Consider cardioselective beta-blockers"
            },
            {
                "condition": "diabetes_type_1",
                "description": "Insulin-dependent diabetes",
                "reason": "May mask hypoglycemic symptoms",
                "severity": "relative",
                "alternatives": "Use with caution, monitor glucose closely"
            }
        ]
    },

    "warfarin": {
        "absolute": [
            {
                "condition": "active_bleeding",
                "description": "Any active hemorrhage",
                "reason": "Anticoagulation increases bleeding risk",
                "severity": "absolute"
            },
            {
                "condition": "pregnancy",
                "description": "Pregnancy (especially first trimester)",
                "reason": "Teratogenic effects, fetal bleeding risk",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "recent_surgery",
                "description": "Major surgery within 72 hours",
                "reason": "Increased bleeding risk",
                "severity": "relative",
                "alternatives": "Consider bridging with heparin"
            },
            {
                "condition": "liver_disease",
                "description": "Severe hepatic impairment",
                "reason": "Reduced synthesis of clotting factors",
                "severity": "relative",
                "alternatives": "Consider DOACs or reduced dosing"
            }
        ]
    },

    # Antibiotics
    "ciprofloxacin": {
        "absolute": [
            {
                "condition": "fluoroquinolone_allergy",
                "description": "Known hypersensitivity to fluoroquinolones",
                "reason": "Risk of severe allergic reaction",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "seizure_disorder",
                "description": "History of epilepsy or seizures",
                "reason": "Lowers seizure threshold",
                "severity": "relative",
                "alternatives": "Use alternative antibiotic class"
            },
            {
                "condition": "tendon_disorders",
                "description": "History of tendinitis or tendon rupture",
                "reason": "Increased risk of tendon complications",
                "severity": "relative",
                "alternatives": "Consider beta-lactam antibiotics"
            },
            {
                "condition": "age_over_65",
                "description": "Elderly patients",
                "reason": "Increased risk of tendon rupture and CNS effects",
                "severity": "relative",
                "alternatives": "Use with caution, consider alternatives"
            }
        ]
    },

    # Psychiatric medications
    "lithium": {
        "absolute": [
            {
                "condition": "severe_renal_impairment",
                "description": "GFR < 30 mL/min/1.73m²",
                "reason": "Lithium is renally eliminated, toxicity risk",
                "severity": "absolute"
            },
            {
                "condition": "severe_cardiac_disease",
                "description": "Unstable cardiac conditions",
                "reason": "Cardiac conduction effects",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "mild_renal_impairment",
                "description": "GFR 30-60 mL/min/1.73m²",
                "reason": "Reduced clearance, need dose adjustment",
                "severity": "relative",
                "alternatives": "Reduce dose, monitor levels closely"
            },
            {
                "condition": "pregnancy",
                "description": "Pregnancy, especially first trimester",
                "reason": "Risk of cardiac malformations (Ebstein's anomaly)",
                "severity": "relative",
                "alternatives": "Consider alternative mood stabilizers"
            }
        ]
    },

    # Pain medications
    "morphine": {
        "absolute": [
            {
                "condition": "respiratory_depression",
                "description": "Significant respiratory compromise",
                "reason": "Opioids depress respiratory drive",
                "severity": "absolute"
            },
            {
                "condition": "paralytic_ileus",
                "description": "Bowel obstruction",
                "reason": "Opioids decrease GI motility",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "copd",
                "description": "Chronic obstructive pulmonary disease",
                "reason": "Risk of respiratory depression",
                "severity": "relative",
                "alternatives": "Use lowest effective dose, monitor closely"
            },
            {
                "condition": "elderly",
                "description": "Age > 65 years",
                "reason": "Increased sensitivity to opioid effects",
                "severity": "relative",
                "alternatives": "Start with lower doses"
            }
        ]
    },

    # NSAIDs
    "ibuprofen": {
        "absolute": [
            {
                "condition": "active_peptic_ulcer",
                "description": "Active gastric or duodenal ulcer",
                "reason": "NSAIDs increase ulcer bleeding risk",
                "severity": "absolute"
            },
            {
                "condition": "severe_heart_failure",
                "description": "NYHA Class IV heart failure",
                "reason": "Fluid retention and reduced renal function",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "hypertension",
                "description": "Uncontrolled high blood pressure",
                "reason": "NSAIDs may increase blood pressure",
                "severity": "relative",
                "alternatives": "Monitor BP, consider acetaminophen"
            },
            {
                "condition": "kidney_disease",
                "description": "Chronic kidney disease",
                "reason": "Risk of further renal impairment",
                "severity": "relative",
                "alternatives": "Use lowest dose, monitor renal function"
            }
        ]
    },

    # Diabetes medications
    "metformin": {
        "absolute": [
            {
                "condition": "severe_renal_impairment",
                "description": "eGFR < 30 mL/min/1.73m²",
                "reason": "Risk of lactic acidosis",
                "severity": "absolute"
            },
            {
                "condition": "metabolic_acidosis",
                "description": "Any form of acute metabolic acidosis",
                "reason": "Risk of worsening lactic acidosis",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "moderate_renal_impairment",
                "description": "eGFR 30-45 mL/min/1.73m²",
                "reason": "Increased risk of lactic acidosis",
                "severity": "relative",
                "alternatives": "Reduce dose, monitor renal function"
            },
            {
                "condition": "contrast_exposure",
                "description": "Scheduled contrast imaging",
                "reason": "Risk of contrast-induced nephropathy",
                "severity": "relative",
                "alternatives": "Hold 48h before and after contrast"
            }
        ]
    },

    # ACE Inhibitors
    "lisinopril": {
        "absolute": [
            {
                "condition": "angioedema_history",
                "description": "Previous ACE inhibitor-induced angioedema",
                "reason": "High risk of recurrent life-threatening angioedema",
                "severity": "absolute"
            },
            {
                "condition": "pregnancy",
                "description": "Pregnancy (second and third trimester)",
                "reason": "Fetal renal dysgenesis and oligohydramnios",
                "severity": "absolute"
            }
        ],
        "relative": [
            {
                "condition": "bilateral_renal_artery_stenosis",
                "description": "Significant bilateral stenosis",
                "reason": "Risk of acute renal failure",
                "severity": "relative",
                "alternatives": "Use with extreme caution, monitor renal function"
            },
            {
                "condition": "hyperkalemia",
                "description": "Serum K+ > 5.5 mEq/L",
                "reason": "ACE inhibitors increase potassium",
                "severity": "relative",
                "alternatives": "Correct hyperkalemia first"
            }
        ]
    }
}

In [ ]:
def check_drug_contra_indications():
  # IMPLEMENT THIS FUNCTION

agent = ReActAgent(
    tools=[check_drug_contra_indications],
    llm=llm,
)

In [ ]:
# test your agent
query = """
I need help evaluating whether to prescribe metoprolol for a patient with hypertension.
Here's the relevant patient history:

PATIENT HISTORY:
================
Name: John Smith
Age: 62 years old
Chief Complaint: Elevated blood pressure readings at home

MEDICAL HISTORY:
- Hypertension diagnosed 3 years ago
- Type 2 diabetes mellitus (well-controlled on metformin)
- Hyperlipidemia on atorvastatin
- No known drug allergies

CURRENT MEDICATIONS:
- Lisinopril 10mg daily
- Metformin 1000mg twice daily
- Atorvastatin 20mg daily

VITAL SIGNS (from today's visit):
- Blood Pressure: 158/92 mmHg
- Heart Rate: 47 bpm (bradycardic)
- Temperature: 98.6°F
- Respiratory Rate: 16/min
- Oxygen Saturation: 98% on room air

PHYSICAL EXAM:
- Regular heart rhythm, no murmurs
- Lungs clear to auscultation
- No peripheral edema
- No signs of heart failure

LAB RESULTS:
- HbA1c: 7.1%
- Creatinine: 1.1 mg/dL (eGFR >60)
- Potassium: 4.2 mEq/L

ASSESSMENT:
Patient's blood pressure remains elevated despite current ACE inhibitor therapy.
Considering adding metoprolol 25mg twice daily for additional BP control.

Can you evaluate the safety of adding metoprolol to this patient's regimen?
"""


await check_step_by_step(agent, query=query)

## Combining both Agents into one
Now it's time to combine both tools into one agent!

In [ ]:
agent = ReActAgent(
    tools=[check_drug_interaction, check_drug_contra_indications],
    llm=llm,
)

In [ ]:
# test your agent
query = """A 68-year-old patient with atrial fibrillation, diabetes type 2, and a history of asthma
is currently taking warfarin 5mg daily and metformin 1000mg twice daily.

The cardiologist wants to add metoprolol 25mg twice daily for better rate control of
the atrial fibrillation, as the patient's heart rate has been consistently above 100 bpm.

Please evaluate this proposed medication addition by checking:

1. Are there any contraindications for metoprolol given this patient's medical conditions
   (atrial fibrillation, diabetes type 2, asthma)?

2. Are there any significant drug interactions between metoprolol and the patient's
   current medications?

3. Based on your findings from both the contraindication and drug interaction checks,
   what is your clinical recommendation? Should metoprolol be prescribed, avoided,
   or used with specific precautions?

Please provide a comprehensive analysis that addresses patient safety concerns and
offers alternative suggestions if needed."""

await check_step_by_step(agent, query=query)